# RSNA Pneumonia Detection — YOLO Pipeline (Colab)

End-to-end пайплайн для обучения YOLO-модели детекции пневмонии
на рентгеновских снимках грудной клетки.

Датасет: [RSNA Pneumonia Detection Challenge](https://www.kaggle.com/c/rsna-pneumonia-detection-challenge) —
26 684 снимка, 9 555 боксов, один класс.

**Репозиторий:** [e-eki/rsna-pneumonia-yolo-pipeline](https://github.com/e-eki/rsna-pneumonia-yolo-pipeline) (ветка `development`)

**Runtime:** `Runtime → Change runtime type → T4 GPU`

**Время полного прогона:** ~4 ч 30 мин (30 эпох, `yolov8s`)

---

## Полный цикл пайплайна:

1. Клонирование репозитория и установка зависимостей
2. Настройка Kaggle API через Colab Secrets
3. Скачивание датасета
4. Подготовка YOLO-датасета с фильтрацией разметки
5. Статистика и визуальные проверки
6. Обучение модели
7. Оценка на валидационной выборке
8. Итоговая сводка
9. Сохранение артефактов на Google Drive (опционально)

## 📚 Документация проекта

- [README](../../README.md) — общее описание и быстрый старт
- [STRUCTURE](../../docs/STRUCTURE.md) — структура проекта и стадии пайплайна
- [ARCHITECTURE](../../docs/ARCHITECTURE.md) — схема потока данных
- [DATA](../../docs/DATA.md) — датасет и фильтрация разметки
- [TRAINING](../../docs/TRAINING.md) — гиперпараметры и аугментации
- [EVALUATION](../../docs/EVALUATION.md) — метрики и как их читать
- [RESULTS](../../docs/RESULTS.md) — метрики обученной модели

## 0. Проверка GPU

**Что входит в проверку:**

- `nvidia-smi` — есть ли физический GPU и сколько памяти.
- `torch.cuda.is_available()` — видит ли PyTorch GPU.
- `torch.cuda.get_device_name(0)` — какая именно карта выдана.

**Ожидаемый результат:** `Tesla T4` и `True`. Если результат `False`
или `CPU` — зайти в `Runtime → Change runtime type → T4 GPU` и
перезапустить ячейку.

In [ ]:
# Информация о GPU: модель, память, драйвер, загрузка
!nvidia-smi

In [ ]:
# Проверка, что PyTorch видит GPU и какой именно
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

## 1. Клонирование репозитория

Скачивает исходный код проекта из GitHub в Colab.

После этого все скрипты пайплайна будут доступны как `python -m src.<script>`.

In [ ]:
# переходим в рабочую папку Colab
%cd /content

# Удаляем старую папку, если она осталась от прошлой сессии
!rm -rf rsna-pneumonia-yolo-pipeline

!git clone -b development https://github.com/e-eki/rsna-pneumonia-yolo-pipeline.git

In [ ]:
# переходим в проект
%cd rsna-pneumonia-yolo-pipeline

# проверка ветки: должно быть development
!git branch --show-current

## 2. Установка зависимостей

Устанавливает Python-пакеты, которые нужны пайплайну: `ultralytics` (YOLO),
`torch`, `opencv-python`, `pandas` и другие — полный список в
`requirements.txt`.

**Если pip просит перезапустить рантайм** (`You must restart the runtime`)
— это нормально после первой установки. `Runtime → Restart runtime`,
потом запустить эту ячейку ещё раз.

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# Проверка: если импорт ultralytics проходит без ошибок, всё установилось корректно.

import ultralytics
print(f"ultralytics {ultralytics.__version__}")

## 3. Настройка Kaggle API

Чтобы скачать датасет, Colab должен авторизоваться на Kaggle. Токен
хранится в Colab Secrets.

**Как получить токен:**

1. Открыть [kaggle.com](https://www.kaggle.com) → войти в аккаунт.
2. `Account` → `Settings` → раздел `API Tokens` → `Generate New Token`.
3. Скопировать значение — оно начинается с `KGAT_`.

**Как добавить в Colab:**

1. Значок 🔑 на левой панели Colab.
2. `Add new secret`.
3. Имя: `KAGGLE_API_TOKEN` (важно — точное совпадение).
4. Значение: вставить токен `KGAT_...`.
5. Включить опцию **Notebook access** — без неё Colab не отдаст секрет.

**Что делает код ниже:**

1. Достаёт `KAGGLE_API_TOKEN` из Colab Secrets и кладёт в переменную
   окружения — так его видит `kaggle` CLI.
2. Проверяет, что CLI действительно видит токен, запросом метаданных
   датасета.


In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
    print("✅ KAGGLE_API_TOKEN загружен из Colab Secrets")
except Exception as e:
    raise RuntimeError(
        "Не удалось получить KAGGLE_API_TOKEN. "
        "Добавьте секрет с этим именем в 🔑 на левой панели Colab."
    ) from e

# Проверка, что CLI видит токен
!kaggle datasets metadata poeticmage/rsna-pneumonia-detection-challenge-train-val

## 4. Скачивание датасета

Запускает `download_data.py` — скачивает датасет с Kaggle и распаковывает
его в `data/raw/rsna_yolo/`.

**Что появится в `data/raw/`:**
```
data/raw/rsna_yolo/
├── images/{train,val}/*.png   # 1024×1024 PNG
└── labels/{train,val}/*.txt   # YOLO-формат: cls xc yc w h
```

In [ ]:
!python -m src.download_data --config configs/config.yaml

In [ ]:
# Проверка, что структура и числа совпадают с ожидаемыми
!echo "train images: $(ls data/raw/rsna_yolo/images/train | wc -l)"
!echo "train labels: $(ls data/raw/rsna_yolo/labels/train | wc -l)"
!echo "train labels с боксами: $(find data/raw/rsna_yolo/labels/train -name '*.txt' -size +0c | wc -l)"
!echo "val images:   $(ls data/raw/rsna_yolo/images/val | wc -l)"
!echo "val labels с боксами: $(find data/raw/rsna_yolo/labels/val -name '*.txt' -size +0c | wc -l)"

## 5. Подготовка датасета

Запускает `prepare_yolo_dataset.py` — главный скрипт подготовки данных.
На выходе получаем чистый YOLO-датасет, готовый к обучению.

**Что делает скрипт:**

1. Проверяет пары `image ↔ label` — каждой картинке должен соответствовать `.txt` с разметкой.
2. Фильтрует «плохие» боксы: микроскопические, слишком вытянутые, выходящие за границы кадра.
3. Изображения с критичной разметкой убирает в `data/samples/rejected/` — чтобы модель не училась «здесь нет очага» на снимке, где он размечен.
4. Конвертирует PNG → JPEG q=92 (в 3 раза быстрее I/O при обучении).
5. Собирает `data/processed/` + `data.yaml` для Ultralytics.

**Куда что сохраняется:**

- `data/processed/images/{train,val}/*.jpg` — чистые изображения
- `data/processed/labels/{train,val}/*.txt` — отфильтрованная разметка
- `data/processed/data.yaml` — конфиг для YOLO
- `data/samples/rejected/rejected_images/` — снимки с критичной разметкой
- `data/samples/rejected/rejected_annotations.csv` — отвергнутые боксы с причинами
- `data/staged/filtering_report.json` — сводка фильтрации

**На что смотреть после запуска:**
- `ls data/processed` — должны быть `images/`, `labels/`, `data.yaml`.
- `rejected_annotations.csv` — сколько боксов отвергнуто и почему

Подробнее про правила фильтрации — в [DATA.md](../../DATA.md) репозитория.

In [ ]:
# Если папки остались от прошлой сессии — удаляем
!rm -rf data/processed data/staged data/samples

!python -m src.prepare_yolo_dataset --config configs/config.yaml

In [ ]:
# Проверка структуры подготовленного датасета

!ls data/processed
!ls data/processed/images/train | head -2
!ls data/processed/labels/train | head -2

In [ ]:
# Данные об отвергнутых изображениях

import pandas as pd
df = pd.read_csv("data/samples/rejected/rejected_annotations.csv")
print(df.to_string())

## 6. Статистика и визуальные проверки

Перед обучением стоит убедиться, что датасет собран корректно: боксы
в правильных местах, разметка не побита, отвергнутые снимки — действительно
мусор. Этот раздел запускает три проверки.

**Что делают три скрипта:**

- **`analyze_annotations`** — считает статистику по разметке и рисует
  графики: сколько боксов на изображение, распределение размеров
  и aspect ratio, баланс positive/negative. Проверяет, что все координаты
  лежат в `[0, 1]`. Результат — в `data/reports/processed_{train,val}/`.
- **`sample_and_draw`** — берёт 50 случайных снимков и отрисовывает
  на них боксы. Плюс собирает контактный лист — сетку превью. Это самый
  быстрый способ глазами убедиться, что разметка «садится» на очаги.
  Результат — в `data/samples/processed/{train,val}/`.
- **`collect_rejected`** — рисует на отвергнутых снимках те боксы,
  из-за которых они были удалены. Красный = minor, оранжевый = critical.
  Результат — в `data/samples/rejected/drawn/`.

**Зачем:**

- Если в статистике видны аномалии (например, микроскопические боксы
  или перекос классов) — их лучше заметить до обучения.
- Если контактный лист показывает боксы «мимо» очагов — значит, где-то
  ошибка в парсинге координат.
- Если в `rejected/drawn/` лежит снимок с нормальным очагом — фильтр
  слишком агрессивный.

Все три скрипта идемпотентны: перезапуск перерисует графики и сэмплы
с нуля.

In [ ]:
# Сбор статистики по разметке: графики и проверки на выход за границы для train/val

!python -m src.analyze_annotations --stage processed --split train
!python -m src.analyze_annotations --stage processed --split val

In [ ]:
# Создание случайной проверочной выборки с отрисованной разметкой для train/val

!python -m src.sample_and_draw --stage processed --split train --num 50 --grid
!python -m src.sample_and_draw --stage processed --split val   --num 50 --grid

In [ ]:
# Визуализация отвергнутых изображений: отрисовка на них отклонённых боксов

!python -m src.collect_rejected --config configs/config.yaml

### 6.1. Просмотр сэмплов (слайдер)

Слайдер листает изображения из папки — вместо того, чтобы открывать
файлы по одному в Colab.

**Что смотреть в разных папках:**

- `data/samples/processed/train` — случайные train-снимки с отрисованными боксами.
- `data/samples/processed/val` — то же для val.
- `data/samples/rejected/drawn` — отвергнутые снимки: красный бокс = minor, оранжевый = critical.

**Перед запуском** поменяйте `FOLDER` в ячейке ниже на нужную папку.

In [ ]:
from pathlib import Path
from IPython.display import display, Image
import ipywidgets as widgets

FOLDER = Path("data/samples/rejected/drawn")  # ← обработанные: processed/train или processed/val
paths = sorted(FOLDER.glob("*.jpg"))

if not paths:
    raise RuntimeError(f"Нет изображений в {FOLDER}")

slider = widgets.IntSlider(
    min=0, max=len(paths) - 1, step=1,
    description=f"0/{len(paths)-1}",
)
out = widgets.Output()

def render(idx):
    with out:
        out.clear_output(wait=True)
        print(paths[idx].name)
        display(Image(str(paths[idx]), width=800))

widgets.interactive_output(render, {"idx": slider})
display(slider, out)

## 7. Обучение

Запускает `train.py` — обучает YOLO на `data/processed/` и сохраняет
артефакты в `runs/train/<run_name>/`.

**Параметры** (модель, эпохи, batch, imgsz, аугментации) — в
`configs/config.yaml`, секция `training`. Подробности — в `TRAINING.md`
репозитория.

**В `runs/train/<run_name>/` появятся:**
`weights/best.pt`, `weights/last.pt`, `results.csv`, `args.yaml`,
`results.png`, `confusion_matrix.png`, `labels.jpg`, примеры батчей.

In [ ]:
!python -m src.train --config configs/config.yaml --run-name 2026-10-02_yolov8s_30ep

## 8. Оценка

Оценка обученной YOLO-модели на валидационной выборке.

Скрипт загружает указанные веса (`--model` обязателен), прогоняет
валидацию на data/processed и сохраняет графики и `predictions.json` в `runs/val/<run_name>/`.

Имя папки результатов выводится из пути к весам:
`runs/train/<run_name>/weights/best.pt → runs/val/<run_name>/`
Это нужно, чтобы папки в `train/` и `val/` были парными и легко сопоставлялись. Можно переопределить через `--run-name`, если нужно другое имя папки.

In [ ]:
!python -m src.evaluate --config configs/config.yaml \
  --model runs/train/2026-10-02_yolov8s_30ep/weights/best.pt

### 8.1. Кривые обучения

Просмотр графиков, которые Ultralytics сохраняет после обучения.
Слайдер по PNG-файлам в `runs/train/<run_name>` : `results.png` (кривые обучения), `confusion_matrix.png` (матрица ошибок), `PR_curve.png` / `F1_curve.png` (precision-recall и F1), `labels.jpg`
(распределение боксов), `train_batch*.jpg` (примеры батчей).

**Как пользоваться:**
Перед запуском ячейки укажите `RUN_NAME` — имя папки из `runs/train/`. Если не указать, ячейка упадёт с ошибкой и покажет список доступных прогонов.

In [ ]:
from pathlib import Path
from IPython.display import display, Image
import ipywidgets as widgets

# ── Укажите имя прогона, который хотите посмотреть ──
# Пример: "2026-10-02_yolov8s_30ep"
RUN_NAME = "2026-10-02_yolov8s_30ep"

if not RUN_NAME:
    raise ValueError(
        "Укажите RUN_NAME — имя папки из runs/train/. "
    )

FOLDER = Path("runs/train") / RUN_NAME
if not FOLDER.exists():
    available = sorted(p.name for p in Path("runs/train").glob("*/"))
    raise FileNotFoundError(
        f"Папка {FOLDER} не найдена. "
        f"Доступные прогоны: {available or 'нет'}"
    )

paths = sorted(FOLDER.glob("*.png"))
if not paths:
    raise RuntimeError(f"Нет PNG в {FOLDER}")

slider = widgets.IntSlider(
    min=0, max=len(paths) - 1, step=1,
    description=f"0/{len(paths)-1}",
)
out = widgets.Output()

def render(idx):
    with out:
        out.clear_output(wait=True)
        print(paths[idx].name)
        display(Image(str(paths[idx]), width=800))

widgets.interactive_output(render, {"idx": slider})
display(slider, out)

## 9. Итоговая сводка по пайплайну

Собирает в одном месте ключевые числа по всему пайплайну: что получилось
на каждом этапе и какие метрики у указанного прогона.

**Что показывает:**

1. **Отчёты по `processed/`** — сколько боксов и изображений попало
   в финальный YOLO-датасет (train и val отдельно). Если есть боксы
   вне `[0, 1]` — покажет сколько.
2. **Отчёт фильтрации** — сколько изображений было, сколько positive
   и negative, сколько снимков ушло в `rejected/`, сколько боксов
   сохранено и отвергнуто (с разбивкой по причинам).
3. **Метрики обучения** — финальные mAP@0.5, mAP@0.5:0.95, precision,
   recall указанного прогона (последняя эпоха из `results.csv`).
4. **Папка оценки** — путь к `runs/val/<RUN_NAME>/` или предупреждение,
   что `evaluate.py` для этого прогона ещё не запускался.

**Зачем:**
Один запуск ячейки — и вся картина по пайплайну перед глазами. Не нужно
открывать `filtering_report.json`, `results.csv` и папки `runs/` вручную.

**Перед запуском** укажите `RUN_NAME` — имя папки из `runs/train/`.
Например, `"2026-10-02_yolov8s_30ep"`. Если не указать — ячейка
упадёт и покажет список доступных прогонов.

In [ ]:
import json
from pathlib import Path
import pandas as pd

# ── Укажите имя прогона, по которому строим сводку ──
# Пример: "2026-10-02_yolov8s_30ep"
RUN_NAME = "2026-10-02_yolov8s_30ep"

if not RUN_NAME:
    raise ValueError(
        "Укажите RUN_NAME — имя папки из runs/train/. "
    )

def fmt(v):
    try:
        return f"{float(v):.4f}"
    except (TypeError, ValueError):
        return str(v)

print("=" * 60)
print("СВОДКА ПО ПАЙПЛАЙНУ")
print("=" * 60)

# 1. Отчёты по processed
for split in ("train", "val"):
    report_path = Path("data/reports") / f"processed_{split}" / "report.json"
    if report_path.exists():
        data = json.load(open(report_path))
        bpi = data.get("boxes_per_image", {})
        print(f"\n[processed/{split}]")
        print(f"  боксов:                 {data.get('num_boxes')}")
        print(f"  изображений с боксами:  {bpi.get('with_boxes')}")
        print(f"  изображений без боксов: {bpi.get('without_boxes')}")
        if "yolo_out_of_bounds" in data:
            print(f"  YOLO out-of-bounds:     {data['yolo_out_of_bounds']}")

# 2. Отчёт фильтрации
filt = Path("data/staged/filtering_report.json")
if filt.exists():
    data = json.load(open(filt))
    print(f"\n[Фильтрация]")
    for s in data.get("splits", []):
        print(f"  [{s['split']}]")
        print(f"    изображений всего:   {s['images_total']}")
        print(f"    positive:            {s['images_positive']}")
        print(f"    negative:            {s['images_negative']}")
        print(f"    dropped_to_rejected: {s['images_dropped_to_rejected']}")
        print(f"    боксов сохранено:    {s['boxes_kept']}")
        print(f"    боксов отвергнуто:   {s['boxes_rejected']}")
        for reason, count in s.get("rejected_by_reason", {}).items():
            if count:
                print(f"      {reason}: {count}")

# 3. Метрики указанного train-прогона
results_csv = Path("runs/train") / RUN_NAME / "results.csv"
if not results_csv.exists():
    available = sorted(p.parent.name for p in Path("runs/train").glob("*/results.csv"))
    raise FileNotFoundError(
        f"{results_csv} не найден. "
        f"Доступные прогоны: {available or 'нет'}"
    )
df = pd.read_csv(results_csv)
last = df.iloc[-1]
print(f"\n[Обучение — {RUN_NAME}]")
print(f"  эпох:         {int(last.get('epoch', -1))}")
print(f"  mAP@0.5:      {fmt(last.get('metrics/mAP50(B)'))}")
print(f"  mAP@0.5:0.95: {fmt(last.get('metrics/mAP50-95(B)'))}")
print(f"  precision:    {fmt(last.get('metrics/precision(B)'))}")
print(f"  recall:       {fmt(last.get('metrics/recall(B)'))}")

# 4. Оценка для этого же прогона
val_dir = Path("runs/val") / RUN_NAME
if val_dir.exists():
    print(f"\n[Оценка — {RUN_NAME}]")
    print(f"  {val_dir}")
else:
    print(f"\n[Оценка — {RUN_NAME}]  не найдена (запустите src.evaluate)")

print("\n" + "=" * 60)

## 10. Сохранение результатов на Google Drive (опционально)

Сохранение `runs/` и `data/reports/` — чтобы после отключения Colab-рантайма остались метрики и логи.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!tar -czf /content/drive/MyDrive/rsna-runs.tar.gz \
     -C /content/rsna-pneumonia-yolo-pipeline runs data/reports data/staged

!ls -lh /content/drive/MyDrive/rsna-runs.tar.gz